# Experiment 8: Multilingual Abstractive Summarization (mT5 vs mBART)

## Lab Questions / Objectives

1. **To implement and evaluate abstractive text summarization using two multilingual Transformer-based sequence-to-sequence models:**
   1. mT5 (Multilingual Text-to-Text Transfer Transformer)
   2. mBART (Multilingual Bidirectional and Auto-Regressive Transformers)
   * *Dataset:* XL-Sum (Cross-Lingual Summarization) dataset (Hindi language subset chosen).
   * *Evaluation Metrics:* ROUGE-1, ROUGE-2, ROUGE-L, and mBERT / BERTScore.
   * *Dataset Link:* https://huggingface.co/datasets/csebuetnlp/xlsum

In [1]:
# ==============================================================================
# EXPERIMENT 8: Multilingual Abstractive Summarization (mT5 vs mBART)
# Dataset: XL-Sum (Hindi) | Metrics: ROUGE-1/2/L + BERTScore (mBERT)
# ==============================================================================

# 1. Install dependencies
!pip install -q transformers datasets evaluate rouge-score bert-score sentencepiece pandas

# 2. Imports
import re
import torch
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          MBart50TokenizerFast, MBartForConditionalGeneration)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 3. Load XL-Sum Hindi test split from the auto-converted parquet branch
print("Loading XL-Sum Dataset (Hindi)...")
PARQUET_URL = ("https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/"
               "refs%2Fconvert%2Fparquet/hindi/test/0000.parquet")
dataset = load_dataset("parquet", data_files={"test": PARQUET_URL}, split="test[:10]")

articles = dataset["text"]
references = dataset["summary"]
print(f"Loaded {len(articles)} samples")

# Whitespace cleanup (recommended by XL-Sum authors)
WHITESPACE_HANDLER = lambda k: re.sub(r"\s+", " ", re.sub(r"\n+", " ", k.strip()))

# Load metrics
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

# ------------------------------------------------------------------------------
# MODEL 1: mT5 (fine-tuned on XL-Sum)
# ------------------------------------------------------------------------------
print("\n--- Generating with mT5 ---")
mt5_name = "csebuetnlp/mT5_multilingual_XLSum"
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_name)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_name).to(device)
mt5_model.eval()

mt5_summaries = []
for i, text in enumerate(articles):
    inputs = mt5_tokenizer(WHITESPACE_HANDLER(text), return_tensors="pt",
                           max_length=512, truncation=True).to(device)
    with torch.no_grad():
        ids = mt5_model.generate(**inputs, max_length=84, num_beams=4,
                                 no_repeat_ngram_size=2, early_stopping=True)
    mt5_summaries.append(mt5_tokenizer.decode(ids[0], skip_special_tokens=True))
    print(f"  mT5 sample {i+1}/{len(articles)} done")

del mt5_model
if device == "cuda":
    torch.cuda.empty_cache()

# ------------------------------------------------------------------------------
# MODEL 2: mBART (facebook/mbart-large-50)
# ------------------------------------------------------------------------------
print("\n--- Generating with mBART ---")
mbart_name = "facebook/mbart-large-50"
mbart_tokenizer = MBart50TokenizerFast.from_pretrained(mbart_name, src_lang="hi_IN", tgt_lang="hi_IN")
mbart_model = MBartForConditionalGeneration.from_pretrained(mbart_name).to(device)
mbart_model.eval()
hi_token_id = mbart_tokenizer.convert_tokens_to_ids("hi_IN")   # replaces removed lang_code_to_id

mbart_summaries = []
for i, text in enumerate(articles):
    inputs = mbart_tokenizer(WHITESPACE_HANDLER(text), return_tensors="pt",
                             max_length=512, truncation=True).to(device)
    with torch.no_grad():
        ids = mbart_model.generate(**inputs, forced_bos_token_id=hi_token_id,
                                   max_length=84, num_beams=4,
                                   no_repeat_ngram_size=2, early_stopping=True)
    mbart_summaries.append(mbart_tokenizer.decode(ids[0], skip_special_tokens=True))
    print(f"  mBART sample {i+1}/{len(articles)} done")

# ------------------------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------------------------
def compute_metrics(predictions, targets):
    # Default ROUGE tokenizer drops non-Latin characters, so split on whitespace for Hindi
    rouge_res = rouge_metric.compute(predictions=predictions, references=targets,
                                     tokenizer=lambda x: x.split())
    # lang="hi" -> uses bert-base-multilingual-cased (mBERT)
    bert_res = bertscore_metric.compute(predictions=predictions, references=targets, lang="hi")
    return {
        "ROUGE-1": round(rouge_res["rouge1"], 4),
        "ROUGE-2": round(rouge_res["rouge2"], 4),
        "ROUGE-L": round(rouge_res["rougeL"], 4),
        "mBERT Score (F1)": round(sum(bert_res["f1"]) / len(bert_res["f1"]), 4),
    }

print("\nComputing evaluation metrics...")
mt5_eval = compute_metrics(mt5_summaries, references)
mbart_eval = compute_metrics(mbart_summaries, references)

comparison_df = pd.DataFrame([mt5_eval, mbart_eval],
                             index=["mT5 (XL-Sum fine-tuned)", "mBART-large-50"])

print("\n" + "=" * 60)
print("EXPERIMENT 8 MODEL COMPARISON SUMMARY")
print("=" * 60)
print(comparison_df)

# Sample outputs
print("\n" + "=" * 60)
print("SAMPLE OUTPUT (Article 1)")
print("=" * 60)
print("Reference :", references[0])
print("mT5       :", mt5_summaries[0])
print("mBART     :", mbart_summaries[0])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using device: cuda
Loading XL-Sum Dataset (Hindi)...


hindi/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.5MB            

hindi/test/0000.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Loaded 10 samples



--- Generating with mT5 ---


... (29 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EXPERIMENT 8 MODEL COMPARISON SUMMARY
                         ROUGE-1  ROUGE-2  ROUGE-L  mBERT Score (F1)
mT5 (XL-Sum fine-tuned)   0.3126   0.1372   0.2621            0.7610
mBART-large-50            0.1963   0.0519   0.1510            0.6997

SAMPLE OUTPUT (Article 1)
Reference : मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्वी प्रज्ञा सिंह ठाकुर को अब पॉलीग्राफ़, ब्रेन मैपिंग और नॉर्को टेस्ट से गुज़रना होगा. ये सारे टेस्ट झूठ पकड़ने के लिए किए जाते हैं.
mT5       : मालेगाँव धमाके के अभियुक्त साध्वी प्रज्ञा सिंह ठाकुर को तीन नवंबर तक के लिए पुलिस हिरासत में रखने का आदेश दिया गया है.
mBART     : नासिक की एक अदालत ने महाराष्ट्र पुलिस के आतंकवाद निरोधक दस्ते (एटीएस) के अनुरोध पर ये आदेश दिया है
